# GNN V1LD — Entraînement complet (dataset entier)

Entraîne le GNN V1LD sur l'intégralité d'un dataset collecté, avec séparation
**train/test par `inst_id`** (aucun nœud d'une même instance n'est à la fois dans
les deux ensembles).

**Dataset supportés** : `train_v1LD` (élagage Lagrangien) ou `train_base` (CP pur).

**Loss** : `L1(λ_GNN)` — borne résiduelle, auto-supervisée, à minimiser.  
**Métrique** : gap relatif global = `(curr_cost + L1_GNN − opt) / opt × 100`.

## 1. Setup

In [8]:
import sys, importlib, time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

REPO = Path("../").resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import gnn.train_gnn_v1ld as T
importlib.reload(T)
from gnn.train_gnn_v1ld import load_dataset, GNN, lagrangian_v1ld

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

## 2. Configuration

In [9]:
# ── Dataset ───────────────────────────────────────────────────────────────
DATASET    = "train_v1LD"   # "train_v1LD" ou "train_base"
N_VAL      = 10

# ── Split train / test ────────────────────────────────────────────────────
TRAIN_FRAC = 0.8
SEED       = 42

# ── Entraînement ──────────────────────────────────────────────────────────
N_EPOCHS   = 10          # 1 epoch = 1 passage complet du train set
BATCH_SIZE = 32          # gradient accumulation : 1 step tous les B graphes
LR         = 1e-3
ETA_MIN    = 1e-5
LOG_EVERY  = 5           # afficher la loss tous les N epochs
SAVE_EVERY = 10          # sauvegarder checkpoint tous les N epochs
CKPT_PATH  = REPO / "gnn" / f"checkpoint_full_v1ld_{DATASET}.pt"

# ── Évaluation intermédiaire (pendant l'entraînement) ────────────────────
EVAL_EVERY       = 10    # évaluer toutes les N epochs (0 = désactivé)
EVAL_SAMPLE_SIZE = 200   # nb graphes test pour l'éval intermédiaire

# ── Évaluation finale ─────────────────────────────────────────────────────
# None = utiliser tout le train set (lent) ; entier = échantillon aléatoire
FINAL_EVAL_N_TRAIN = 1000

device = torch.device("cpu")
print(f"Dataset : {DATASET}  |  device : {device}")

Dataset : train_v1LD  |  device : cpu


## 3. Chargement et split train/test

In [10]:
_suffix   = "_gbab" if DATASET == "train_v1LD" else ""
DATA_DIR  = REPO / "data" / DATASET
inst_file  = DATA_DIR / f"instances_n{N_VAL}{_suffix}.txt"
nodes_file = DATA_DIR / f"nodes_n{N_VAL}{_suffix}.txt"

print(f"Instances : {inst_file}")
print(f"Nodes     : {nodes_file}")
assert inst_file.exists(),  f"Introuvable : {inst_file}"
assert nodes_file.exists(), f"Introuvable : {nodes_file}"

print("\nChargement...")
t0 = time.time()
all_graphs = load_dataset(inst_file, nodes_file)
print(f"  {len(all_graphs)} graphes chargés en {time.time()-t0:.1f}s")

# ── Split par inst_id ─────────────────────────────────────────────────────
inst_ids = np.array(sorted(set(int(g.inst_id.item()) for g in all_graphs)))
rng = np.random.default_rng(SEED)
rng.shuffle(inst_ids)
n_train_inst = int(len(inst_ids) * TRAIN_FRAC)
train_ids = set(inst_ids[:n_train_inst].tolist())
test_ids  = set(inst_ids[n_train_inst:].tolist())

train_graphs = [g for g in all_graphs if int(g.inst_id.item()) in train_ids]
test_graphs  = [g for g in all_graphs if int(g.inst_id.item()) in test_ids]

assert len(train_ids & test_ids) == 0, "Fuite train/test !"

print(f"\nSplit (seed={SEED}, ratio={TRAIN_FRAC}) :")
print(f"  train : {len(train_ids):4d} instances  →  {len(train_graphs):6d} graphes")
print(f"  test  : {len(test_ids):4d} instances  →  {len(test_graphs):6d} graphes")
print(f"  noeuds/instance (moyen) : {len(all_graphs)/len(inst_ids):.1f}")

Instances : /Users/roman.lendormy/Documents/Cours/X25 - 3A/Stage/Learning_bounds_CCCP/data/train_v1LD/instances_n10_gbab.txt
Nodes     : /Users/roman.lendormy/Documents/Cours/X25 - 3A/Stage/Learning_bounds_CCCP/data/train_v1LD/nodes_n10_gbab.txt

Chargement...
[load_dataset] 27158 graphes chargés, 2199 ignorés
  27158 graphes chargés en 13.4s

Split (seed=42, ratio=0.8) :
  train :  227 instances  →   21700 graphes
  test  :   57 instances  →    5458 graphes
  noeuds/instance (moyen) : 95.6


## 4. Benchmark — estimation du temps d'entraînement

In [11]:
N_BENCH   = 30
_m        = train_graphs[0].m
_tmp      = GNN(m=_m).to(device)
_tmp.eval()

t0 = time.time()
with torch.no_grad():
    for g in train_graphs[:N_BENCH]:
        lam = _tmp(g.to(device))
        _   = lagrangian_v1ld(lam, g.to(device)).item()
t_per_g = (time.time() - t0) / N_BENCH
del _tmp

t_epoch   = t_per_g * len(train_graphs)
t_total   = t_epoch * N_EPOCHS
print(f"Temps par graphe (inférence + L1) : {t_per_g*1000:.1f} ms")
print(f"Temps par epoch  (train complet)  : {t_epoch:.0f} s  ({t_epoch/60:.1f} min)")
print(f"Temps total estimé ({N_EPOCHS} epochs)  : {t_total/3600:.2f} h")
print()
print("Note : pendant l'entraînement, backward() est fait à chaque graphe")
print("→ attendre ≈2× plus longtemps que l'inférence seule.")

Temps par graphe (inférence + L1) : 20.0 ms
Temps par epoch  (train complet)  : 433 s  (7.2 min)
Temps total estimé (10 epochs)  : 1.20 h

Note : pendant l'entraînement, backward() est fait à chaque graphe
→ attendre ≈2× plus longtemps que l'inférence seule.


## 5. Pré-calcul de la baseline L1(λ=0)

In [12]:
def compute_l1_zero(graphs, device):
    """Calcule L1(λ=0) pour chaque graphe (pas de backward)."""
    l1_0s = []
    with torch.no_grad():
        for g in graphs:
            g   = g.to(device)
            lam = torch.zeros(g.N_z * g.n_g, device=device)
            l1_0s.append(lagrangian_v1ld(lam, g).item())
    return np.array(l1_0s)

# Test set complet
print(f"L1(λ=0) sur le test set ({len(test_graphs)} graphes)...")
t0 = time.time()
l1_0_test = compute_l1_zero(test_graphs, device)
print(f"  {time.time()-t0:.1f}s  —  mean={np.mean(l1_0_test):.2f}  "
      f"median={np.median(l1_0_test):.2f}")

# Échantillon train pour l'évaluation finale
_rng2 = np.random.default_rng(SEED + 1)
_n    = min(FINAL_EVAL_N_TRAIN or len(train_graphs), len(train_graphs))
eval_train_idx    = _rng2.choice(len(train_graphs), _n, replace=False)
eval_train_sample = [train_graphs[i] for i in eval_train_idx]

print(f"\nL1(λ=0) sur {_n} graphes train...")
t0 = time.time()
l1_0_train_sample = compute_l1_zero(eval_train_sample, device)
print(f"  {time.time()-t0:.1f}s  —  mean={np.mean(l1_0_train_sample):.2f}")

L1(λ=0) sur le test set (5458 graphes)...
  40.1s  —  mean=1408.46  median=1308.58

L1(λ=0) sur 1000 graphes train...
  7.2s  —  mean=1441.76


## 6. Entraînement

In [13]:
def eval_gap(graphs, l1_0s, model, device):
    """
    Retourne deux arrays de même longueur :
      gaps[i] = (curr_cost + L1_GNN - opt) / opt × 100   (gap global, ≥0 si borne valide)
      imps[i] = (L1_0 - L1_GNN) / L1_0 × 100            (amélioration vs λ=0, >0 = mieux)
    """
    model.eval()
    gaps, imps = [], []
    with torch.no_grad():
        for g, l0 in zip(graphs, l1_0s):
            g      = g.to(device)
            lam    = model(g)
            l1_gnn = lagrangian_v1ld(lam, g).item()
            curr_cost = g.context[3].item()   # index 3 : curr_cost
            opt       = g.opt.item()
            global_bd = curr_cost + l1_gnn
            gaps.append((global_bd - opt) / max(abs(opt), 1e-9) * 100)
            imps.append((l0 - l1_gnn)   / max(abs(l0),  1e-9) * 100)
    return np.array(gaps), np.array(imps)

In [ ]:
torch.manual_seed(SEED)
model     = GNN(m=train_graphs[0].m).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=ETA_MIN)

print(f"Modèle : {sum(p.numel() for p in model.parameters())} paramètres")
print(f"Entraînement : {N_EPOCHS} epochs × {len(train_graphs)} graphes, "
      f"batch={BATCH_SIZE}\n")

history_loss     = []   # (epoch, mean_loss)
history_test_gap = []   # (epoch, mean_gap, median_gap)
indices = np.arange(len(train_graphs))
t_start = time.time()

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    np.random.shuffle(indices)
    epoch_loss = 0.0
    optimizer.zero_grad()

    for step_i, idx in enumerate(indices):
        g    = train_graphs[idx].to(device)
        lam  = model(g)
        loss = lagrangian_v1ld(lam, g)
        # Diviser par BATCH_SIZE pour que le gradient corresponde à une moyenne
        (loss / BATCH_SIZE).backward()
        epoch_loss += loss.item()

        is_last = step_i == len(indices) - 1
        if (step_i + 1) % BATCH_SIZE == 0 or is_last:
            optimizer.step()
            optimizer.zero_grad()

    scheduler.step()
    mean_loss = epoch_loss / len(train_graphs)
    history_loss.append((epoch, mean_loss))

    if epoch % LOG_EVERY == 0 or epoch == 1:
        elapsed   = time.time() - t_start
        remaining = elapsed / epoch * (N_EPOCHS - epoch)
        print(f"epoch {epoch:4d}/{N_EPOCHS}  loss={mean_loss:9.2f}  "
              f"lr={optimizer.param_groups[0]['lr']:.1e}  "
              f"elapsed={elapsed/60:.1f}min  reste~{remaining/60:.0f}min")

    if EVAL_EVERY and epoch % EVAL_EVERY == 0:
        _sample = test_graphs[:EVAL_SAMPLE_SIZE]
        _l0     = l1_0_test[:EVAL_SAMPLE_SIZE]
        g_arr, i_arr = eval_gap(_sample, _l0, model, device)
        history_test_gap.append((epoch, float(np.mean(g_arr)),
                                          float(np.median(g_arr))))
        print(f"  eval test  gap mean={np.mean(g_arr):+.1f}%  "
              f"median={np.median(g_arr):+.1f}%  "
              f"imp_L0 mean={np.mean(i_arr):+.1f}%")
        model.train()

    if SAVE_EVERY and epoch % SAVE_EVERY == 0:
        torch.save({"epoch": epoch, "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "cfg": {"DATASET": DATASET, "N_VAL": N_VAL,
                            "LR": LR, "BATCH_SIZE": BATCH_SIZE}},
                   CKPT_PATH)

total_time = time.time() - t_start
print(f"\nEntraînement terminé en {total_time/60:.1f} min")
torch.save({"epoch": N_EPOCHS, "model": model.state_dict(),
            "cfg": {"DATASET": DATASET, "N_VAL": N_VAL}}, CKPT_PATH)
print(f"Checkpoint final : {CKPT_PATH}")

Modèle : 176449 paramètres
Entraînement : 10 epochs × 21700 graphes, batch=32

epoch    1/10  loss=  1294.23  lr=9.8e-04  elapsed=62.5min  reste~562min


## 7. Évaluation finale du gap relatif

In [ ]:
model.eval()

print(f"Éval test  ({len(test_graphs)} graphes)...")
t0 = time.time()
gaps_test, imps_test = eval_gap(test_graphs, l1_0_test, model, device)
print(f"  {time.time()-t0:.1f}s")

print(f"Éval train ({len(eval_train_sample)} graphes, échantillon)...")
t0 = time.time()
gaps_train, imps_train = eval_gap(eval_train_sample, l1_0_train_sample, model, device)
print(f"  {time.time()-t0:.1f}s")

print()
print(f"{'':6}  {'mean gap':>10}  {'median gap':>11}  {'p90 gap':>9}  "
      f"{'max gap':>9}  {'imp L0 mean':>12}")
print("─" * 65)
for label, gaps, imps in [("TRAIN", gaps_train, imps_train),
                           ("TEST ", gaps_test,  imps_test)]:
    print(f"{label}  {np.mean(gaps):+9.2f}%  {np.median(gaps):+10.2f}%  "
          f"{np.percentile(gaps,90):+8.2f}%  {np.max(gaps):+8.2f}%  "
          f"{np.mean(imps):+11.2f}%")

print()
print("Rappel : gap ≥ 0  ⟺  borne GNN valide (≥ opt)")
print(f"  fraction valide  train : {(gaps_train >= -0.1).mean()*100:.1f}%")
print(f"  fraction valide  test  : {(gaps_test  >= -0.1).mean()*100:.1f}%")

## 8. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── A : courbe de convergence ─────────────────────────────────────────────
ax  = axes[0]
ax2 = ax.twinx()
ep_h = [h[0] for h in history_loss]
l1_h = [h[1] for h in history_loss]
W    = max(1, len(l1_h) // 15)
l1_sm = np.convolve(l1_h, np.ones(W)/W, mode="valid")
ax.plot(ep_h, l1_h, alpha=0.25, color="steelblue", lw=0.8)
ax.plot(ep_h[W-1:], l1_sm, color="steelblue", lw=2, label="loss train (lissée)")
ax.set_xlabel("Epoch"); ax.set_ylabel("L1(λ_GNN) moyen", color="steelblue")
ax.tick_params(axis="y", labelcolor="steelblue")
if history_test_gap:
    ep_tg = [h[0] for h in history_test_gap]
    mg_tg = [h[1] for h in history_test_gap]
    ax2.plot(ep_tg, mg_tg, "r--o", ms=5, lw=1.5, label="gap test moyen (%)")
    ax2.set_ylabel("gap test moyen (%)", color="red")
    ax2.tick_params(axis="y", labelcolor="red")
    ax2.legend(loc="upper right", fontsize=8)
ax.set_title("Convergence")
ax.grid(True, alpha=0.3)

# ── B : boxplot du gap global ─────────────────────────────────────────────
ax = axes[1]
bplot = ax.boxplot(
    [gaps_train, gaps_test],
    labels=[f"Train\n({len(eval_train_sample)} g)", f"Test\n({len(test_graphs)} g)"],
    patch_artist=True, notch=False,
    boxprops   = dict(facecolor="steelblue", alpha=0.45),
    medianprops= dict(color="red", lw=2),
    flierprops = dict(marker=".", ms=3, alpha=0.4),
)
ax.axhline(0, color="green", ls="--", lw=1.5, label="opt (gap=0)")
ax.set_ylabel("(bound_global − opt) / opt  ×100 (%)")
ax.set_title("Gap relatif (≥0 = borne valide)")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# ── C : amélioration GNN vs L1(λ=0) ──────────────────────────────────────
ax = axes[2]
all_imps = np.concatenate([imps_train, imps_test])
lo, hi   = np.percentile(all_imps, 1), np.percentile(all_imps, 99)
bins = np.linspace(lo, hi, 45)
ax.hist(imps_train, bins=bins, alpha=0.55, color="steelblue",
        label=f"Train  μ={np.mean(imps_train):+.1f}%", density=True)
ax.hist(imps_test,  bins=bins, alpha=0.55, color="tomato",
        label=f"Test   μ={np.mean(imps_test):+.1f}%",  density=True)
ax.axvline(0, color="black", ls="--", lw=1.2, label="λ=0 (baseline)")
ax.set_xlabel("(L1₀ − L1_GNN) / L1₀  ×100 (%)\n> 0 = GNN améliore la borne")
ax.set_ylabel("Densité")
ax.set_title("Amélioration vs L1(λ=0)")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle(
    f"GNN V1LD — Entraînement complet  [{DATASET}, n={N_VAL}, "
    f"train={len(train_graphs)} / test={len(test_graphs)} graphes]",
    fontsize=12, y=1.01
)
plt.tight_layout()
fig_path = REPO / "gnn" / f"full_train_v1ld_{DATASET}.png"
plt.savefig(fig_path, bbox_inches="tight")
plt.show()
print(f"Figure sauvegardée : {fig_path.name}")

## 9. (Optionnel) Charger un checkpoint et reprendre l'évaluation

In [ ]:
# Décommenter pour charger un checkpoint existant sans réentraîner
#
# ckpt = torch.load(CKPT_PATH, map_location=device)
# model = GNN(m=train_graphs[0].m).to(device)
# model.load_state_dict(ckpt["model"])
# print(f"Checkpoint chargé — epoch {ckpt['epoch']}")
pass